# Final dense/coherency CYGNSS L1 OmF figures

Single-experiment notebook for the final paired CYGNSS L1 thinning/coherency test, `dense075_coh05`, using the 2020-2021 paired OL and DA stats staged under `output/thinning_expts`.

Figures follow the CYGNSS L1 AZ convention:

- rows: CYGL1, CYGL3, ASCAT, SMOS, SMAP
- map columns: matching OL, DA, `(DA - OL) / OL * 100`
- negative percent differences mean the DA run has lower OmF residuals than the paired OL baseline


In [ ]:
from __future__ import annotations

import math
import pickle
import struct
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm, Normalize, TwoSlopeNorm
from netCDF4 import Dataset
from pyproj import CRS, Transformer
from IPython.display import display

REPO = Path.cwd()
if REPO.name != "geosldas-analysis":
    REPO = Path("/Users/amfox/Desktop/geosldas-analysis")
PROJECT = REPO / "projects" / "CYGNSS_L1_AZ"
STATS = PROJECT / "output" / "thinning_expts"
OUT = STATS / "figures"
TILECOORD = PROJECT / "OLv8_M36_all_sensors_AZ_describe" / "OLv8_M36_all_sensors_AZ.ldas_tilecoord.bin"
EXAMPLE_OBS = PROJECT / "example_obs" / "M06"
OUT.mkdir(parents=True, exist_ok=True)

EXPERIMENT_KEY = "dense075_coh05"
EXPERIMENT_LABEL = "Final dense 0.75 / coherency 0.5"
MONTH_PERIOD = "202001_202112"
FILE_PERIOD = "20200101_20211231"
PERIOD_LABEL = "January 2020-December 2021"

NMIN = 10
LAT_BOUNDARY = 37.5
MAP_EXTENT = (-118.6, -105.2, 28.6, 40.4)
PCT_LIMIT = 30.0

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [ ]:
@dataclass(frozen=True)
class Group:
    name: str
    indices: tuple[int, ...]
    units: str
    color: str


GROUPS = (
    Group("CYGL1", (12,), "dB", "#7b3294"),
    Group("CYGL3", (11,), "m$^3$ m$^{-3}$", "#d95f02"),
    Group("ASCAT", (8, 9, 10), "m$^3$ m$^{-3}$", "#1b9e77"),
    Group("SMOS", (0, 1, 2, 3), "K", "#7570b3"),
    Group("SMAP", (4, 5, 6, 7), "K", "#666666"),
)


@dataclass(frozen=True)
class ExperimentFiles:
    ol_nc4: Path
    da_nc4: Path
    ol_pkl: Path
    da_pkl: Path


FILES = ExperimentFiles(
    ol_nc4=STATS / f"temporal_stats_OL_paired_monitor_xmask_{EXPERIMENT_KEY}_{FILE_PERIOD}.nc4",
    da_nc4=STATS / f"temporal_stats_DA_paired_{EXPERIMENT_KEY}_{FILE_PERIOD}.nc4",
    ol_pkl=STATS / f"spatial_stats_OL_paired_monitor_xmask_{EXPERIMENT_KEY}_{MONTH_PERIOD}.pkl",
    da_pkl=STATS / f"spatial_stats_DA_paired_{EXPERIMENT_KEY}_{MONTH_PERIOD}.pkl",
)

for label, path in FILES.__dict__.items():
    print(f"{label:7s} {path.relative_to(PROJECT)}  exists={path.exists()}")


In [ ]:
def read_exact(stream, nbytes: int) -> bytes:
    data = stream.read(nbytes)
    if len(data) != nbytes:
        raise EOFError(f"expected {nbytes} bytes, got {len(data)}")
    return data


def read_tilecoord(path: Path) -> dict[str, np.ndarray]:
    int_fields = {"tile_id", "typ", "pfaf", "i_indg", "j_indg"}
    fields = [
        "tile_id", "typ", "pfaf", "com_lon", "com_lat", "min_lon", "max_lon",
        "min_lat", "max_lat", "i_indg", "j_indg", "frac_cell", "frac_pfaf", "area", "elev",
    ]
    out = {}
    with path.open("rb") as stream:
        tag = struct.unpack("<i", read_exact(stream, 4))[0]
        if tag != 4:
            raise ValueError(f"unexpected N_tile record tag {tag}")
        n_tile = struct.unpack("<i", read_exact(stream, 4))[0]
        if struct.unpack("<i", read_exact(stream, 4))[0] != tag:
            raise ValueError("N_tile record tags do not match")
        out["N_tile"] = np.asarray(n_tile, dtype=np.int32)
        for field in fields:
            dtype = np.dtype("<i4") if field in int_fields else np.dtype("<f4")
            expected = n_tile * dtype.itemsize
            tag = struct.unpack("<i", read_exact(stream, 4))[0]
            if tag != expected:
                raise ValueError(f"unexpected record tag for {field}: {tag}, expected {expected}")
            out[field] = np.frombuffer(read_exact(stream, expected), dtype=dtype).copy()
            if struct.unpack("<i", read_exact(stream, 4))[0] != tag:
                raise ValueError(f"record tags do not match for {field}")
    return out


def read_nc(path: Path, names=("OmF_stdv", "OmF_mean", "O_mean", "F_mean", "N_data")) -> dict[str, np.ndarray]:
    with Dataset(path) as ds:
        return {
            name: np.ma.filled(ds.variables[name][:], np.nan).astype(float)
            for name in names
        }


def read_monthly(path: Path) -> dict[str, np.ndarray]:
    with path.open("rb") as stream:
        raw = pickle.load(stream)
    source = raw.get("monthly", raw) if isinstance(raw, dict) else raw
    data = {
        key: np.asarray(value, dtype=float) if key != "date_vec" else value
        for key, value in source.items()
    }
    months = []
    for value in data.get("date_vec", []):
        if isinstance(value, datetime):
            months.append(value)
        elif isinstance(value, np.datetime64):
            months.append(value.astype("datetime64[D]").astype(datetime))
        else:
            months.append(datetime.strptime(str(value)[:6], "%Y%m"))
    if not months:
        n_months = data["N_data"].shape[0]
        months = [datetime(2020 + (i // 12), (i % 12) + 1, 1) for i in range(n_months)]
    data["months"] = np.asarray(months)
    return data


def weighted_species(values: np.ndarray, counts: np.ndarray, group: Group, nmin: int = NMIN) -> np.ndarray:
    idx = list(group.indices)
    group_values = values[..., idx]
    group_counts = counts[..., idx]
    valid = np.isfinite(group_values) & np.isfinite(group_counts) & (group_counts >= nmin)
    numerator = np.where(valid, group_values * group_counts, 0.0).sum(axis=-1)
    denominator = np.where(valid, group_counts, 0.0).sum(axis=-1)
    return np.divide(numerator, denominator, out=np.full(denominator.shape, np.nan), where=denominator > 0)


def group_values(data: dict[str, np.ndarray], variable: str, group: Group) -> np.ndarray:
    return weighted_species(data[variable], data["N_data"], group)


def percent_diff(da: np.ndarray, ol: np.ndarray) -> np.ndarray:
    return 100.0 * (da - ol) / ol


def norm_from_values(values, lower=2, upper=98):
    finite_parts = [np.ravel(v[np.isfinite(v)]) for v in values if np.any(np.isfinite(v))]
    if not finite_parts:
        return Normalize(0, 1)
    finite = np.concatenate(finite_parts)
    vmin, vmax = np.nanpercentile(finite, (lower, upper))
    if math.isclose(vmin, vmax):
        delta = max(abs(vmin) * 0.05, 1.0e-6)
        vmin -= delta
        vmax += delta
    return Normalize(vmin=vmin, vmax=vmax)


def signed_norm(values, limit=None, percentile=98):
    if limit is None:
        finite_parts = [np.ravel(np.abs(v[np.isfinite(v)])) for v in values if np.any(np.isfinite(v))]
        finite = np.concatenate(finite_parts) if finite_parts else np.asarray([1.0])
        limit = float(np.nanpercentile(finite, percentile))
    limit = max(float(limit), 1.0e-12)
    return TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)


In [ ]:
tilecoord = read_tilecoord(TILECOORD)
lon = np.asarray(tilecoord["com_lon"], dtype=float)
lat = np.asarray(tilecoord["com_lat"], dtype=float)
area = np.asarray(tilecoord["area"], dtype=float)
lat_mask = lat < LAT_BOUNDARY

spatial = {"ol": read_nc(FILES.ol_nc4), "da": read_nc(FILES.da_nc4)}
monthly = {"ol": read_monthly(FILES.ol_pkl), "da": read_monthly(FILES.da_pkl)}

print("tilecoord tiles:", lon.size)
print("spatial arrays:", spatial["ol"]["OmF_stdv"].shape, spatial["da"]["OmF_stdv"].shape)
print("monthly arrays:", monthly["ol"]["OmF_stdv"].shape, monthly["da"]["OmF_stdv"].shape)
print("months:", monthly["ol"]["months"][0].strftime("%Y-%m"), "to", monthly["ol"]["months"][-1].strftime("%Y-%m"))
print(f"tiles with center latitude < {LAT_BOUNDARY}N:", int(lat_mask.sum()))


In [ ]:
def base_map(ax):
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.94", zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="white", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.45, edgecolor="0.25")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.35, edgecolor="0.35")
    states = cfeature.NaturalEarthFeature(
        "cultural", "admin_1_states_provinces_lakes", "50m", facecolor="none"
    )
    ax.add_feature(states, lw=0.35, edgecolor="0.45")


def scatter_tiles(ax, values, cmap, norm):
    valid = np.isfinite(values)
    return ax.scatter(
        lon[valid], lat[valid], c=values[valid], s=19, marker="s", linewidths=0,
        cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), zorder=2,
    )


def area_weighted_mean(values: np.ndarray, mask: np.ndarray | None = None) -> float:
    valid = np.isfinite(values) & np.isfinite(area) & (area > 0)
    if mask is not None:
        valid &= mask
    if not np.any(valid):
        return np.nan
    return float(np.average(values[valid], weights=area[valid]))


rows = []
for group in GROUPS:
    ol = group_values(spatial["ol"], "OmF_stdv", group)
    da = group_values(spatial["da"], "OmF_stdv", group)
    pct = percent_diff(da, ol)
    rows.append((group, ol, da, pct))

summary_rows = []
for group, ol, da, pct in rows:
    valid = np.isfinite(pct)
    summary_rows.append({
        "experiment": EXPERIMENT_KEY,
        "system": group.name,
        "common_tiles": int(valid.sum()),
        "tile_mean_percent_da_minus_ol": float(np.nanmean(pct)),
        "tile_median_percent_da_minus_ol": float(np.nanmedian(pct)),
        "area_weighted_percent_da_minus_ol": area_weighted_mean(pct),
        f"area_weighted_percent_da_minus_ol_lat_lt_{str(LAT_BOUNDARY).replace('.', 'p')}": area_weighted_mean(pct, lat_mask),
        "fraction_tiles_improved_da_lt_ol": float(np.mean(pct[valid] < 0)),
    })

summary = pd.DataFrame(summary_rows)
summary_path = OUT / f"{EXPERIMENT_KEY}_omf_stdv_percent_map_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)
print("wrote", summary_path.relative_to(PROJECT))

fig = plt.figure(figsize=(13.2, 14.8))
percent_norm = signed_norm([row[3] for row in rows], limit=PCT_LIMIT)
for row_index, (group, ol, da, pct) in enumerate(rows):
    value_norm = norm_from_values([ol, da])
    for col_index, (values, title, cmap, norm) in enumerate((
        (ol, "OL OmF StDev", "viridis", value_norm),
        (da, "DA OmF StDev", "viridis", value_norm),
        (pct, "(DA - OL) / OL (%)", "RdBu_r", percent_norm),
    )):
        ax = fig.add_subplot(5, 3, row_index * 3 + col_index + 1, projection=ccrs.PlateCarree())
        base_map(ax)
        image = scatter_tiles(ax, values, cmap, norm)
        if row_index == 0:
            ax.set_title(title, fontweight="bold", pad=8)
        if col_index == 0:
            ax.text(-0.07, 0.5, group.name, transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontweight="bold", fontsize=11)
        if col_index == 2:
            mean_value = summary.loc[summary.system == group.name, f"area_weighted_percent_da_minus_ol_lat_lt_{str(LAT_BOUNDARY).replace('.', 'p')}"].iloc[0]
            ax.text(0.02, 0.03, f"area-wtd <{LAT_BOUNDARY}N: {mean_value:+.2f}%",
                    transform=ax.transAxes, fontsize=8,
                    bbox={"facecolor": "white", "edgecolor": "0.7", "alpha": 0.88, "pad": 2})
        ax.set_xticks([])
        ax.set_yticks([])
        label = "%" if col_index == 2 else group.units
        cb = fig.colorbar(image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.035)
        cb.set_label(label, fontsize=8)
        cb.ax.tick_params(labelsize=7)

fig.suptitle(f"{EXPERIMENT_LABEL}: OmF standard deviation maps, {PERIOD_LABEL}", y=0.995, fontsize=14)
fig.subplots_adjust(top=0.965, bottom=0.025, left=0.055, right=0.985, hspace=0.25, wspace=0.08)
map_path = OUT / f"{EXPERIMENT_KEY}_omf_stdv_maps_5x3.png"
fig.savefig(map_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", map_path.relative_to(PROJECT))


In [ ]:
def aggregate_monthly(data: dict[str, np.ndarray], variable: str, group: Group) -> np.ndarray:
    return group_values(data, variable, group)


def setup_month_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.grid(True, lw=0.4, alpha=0.35)


def plot_monthly_set(variable: str, figure_label: str, suffix: str):
    months = monthly["ol"]["months"]
    fig, axes = plt.subplots(5, 2, figsize=(13.0, 12.6), sharex=True)
    summary_rows = []

    for row_index, group in enumerate(GROUPS):
        left = axes[row_index, 0]
        right = axes[row_index, 1]
        ol = aggregate_monthly(monthly["ol"], variable, group)
        da = aggregate_monthly(monthly["da"], variable, group)
        pct = percent_diff(da, ol)

        left.plot(months, ol, "--o", color="0.35", lw=1.45, ms=3.3, label="matching OL" if row_index == 0 else None)
        left.plot(months, da, "-o", color=group.color, lw=1.9, ms=3.6, label="DA" if row_index == 0 else None)
        right.plot(months, pct, "-o", color=group.color, lw=1.9, ms=3.6)
        right.axhline(0, color="0.25", lw=0.8)

        left.set_ylabel(f"{group.name}\n{group.units}", fontweight="bold")
        right.set_ylabel("%")
        setup_month_axis(left)
        setup_month_axis(right)
        if row_index == 0:
            left.set_title(f"{figure_label}: matching OL and DA", fontweight="bold")
            right.set_title("Percent difference from matching OL", fontweight="bold")
        if row_index == len(GROUPS) - 1:
            left.set_xlabel("month")
            right.set_xlabel("month")
        for axis in (left, right):
            for label in axis.get_xticklabels():
                label.set_rotation(35)
                label.set_ha("right")

        valid = np.isfinite(pct)
        summary_rows.append({
            "experiment": EXPERIMENT_KEY,
            "variable": variable,
            "system": group.name,
            "monthly_mean_ol": float(np.nanmean(ol)),
            "monthly_mean_da": float(np.nanmean(da)),
            "monthly_mean_percent_da_minus_ol": float(np.nanmean(pct)),
            "monthly_median_percent_da_minus_ol": float(np.nanmedian(pct)),
            "months_improved_da_lt_ol": int(np.sum(pct[valid] < 0)),
            "months_valid": int(valid.sum()),
        })

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle(
        f"{EXPERIMENT_LABEL}: monthly {figure_label}, {PERIOD_LABEL}\n"
        "Percent difference uses 100 x (DA - OL) / OL; negative = lower DA residual",
        y=1.025,
        fontsize=14,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path = OUT / f"{EXPERIMENT_KEY}_monthly_{suffix}_5x2.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")

    table = pd.DataFrame(summary_rows)
    table_path = OUT / f"{EXPERIMENT_KEY}_monthly_{suffix}_summary.csv"
    table.to_csv(table_path, index=False)
    return fig, path, table, table_path

monthly_outputs = []
for variable, label, suffix in (("OmF_stdv", "OmF StDev", "omf_stdv"), ("OmF_mean", "OmF mean", "omf_mean")):
    fig, path, table, table_path = plot_monthly_set(variable, label, suffix)
    monthly_outputs.append((path, table_path))
    display(table)
    plt.show()
    plt.close(fig)
    print("wrote", path.relative_to(PROJECT))
    print("wrote", table_path.relative_to(PROJECT))


In [ ]:
def plot_observation_input_maps():
    fields = (
        ("N_data", "Number of obs", "magma", None, "count"),
        ("O_mean", "Mean obs", "viridis", None, "auto"),
        ("F_mean", "Mean forecast", "viridis", None, "auto"),
    )

    values_by_group = []
    summary_rows = []
    for group in GROUPS:
        group_values_for_row = []
        for variable, title, cmap, _, label in fields:
            source = spatial["da"] if variable == "N_data" else spatial["ol"]
            values = group_values(source, variable, group)
            group_values_for_row.append(values)
            summary_rows.append({
                "experiment": EXPERIMENT_KEY,
                "system": group.name,
                "variable": variable,
                "finite_tiles": int(np.isfinite(values).sum()),
                "tile_mean": float(np.nanmean(values)),
                "tile_median": float(np.nanmedian(values)),
                "area_weighted_mean": area_weighted_mean(values),
                f"area_weighted_mean_lat_lt_{str(LAT_BOUNDARY).replace('.', 'p')}": area_weighted_mean(values, lat_mask),
            })
        values_by_group.append((group, group_values_for_row))

    fig = plt.figure(figsize=(13.2, 14.8))
    image_by_col = {}
    for row_index, (group, row_values) in enumerate(values_by_group):
        for col_index, ((variable, title, cmap, _, label), values) in enumerate(zip(fields, row_values)):
            if variable == "N_data":
                norm = norm_from_values([values], lower=1, upper=99)
            else:
                pair = [
                    group_values(spatial["ol"], variable, group),
                    group_values(spatial["da"], variable, group),
                ]
                norm = norm_from_values(pair)
            ax = fig.add_subplot(5, 3, row_index * 3 + col_index + 1, projection=ccrs.PlateCarree())
            base_map(ax)
            image = scatter_tiles(ax, values, cmap, norm)
            image_by_col[col_index] = image
            if row_index == 0:
                ax.set_title(title, fontweight="bold", pad=8)
            if col_index == 0:
                ax.text(-0.07, 0.5, group.name, transform=ax.transAxes, rotation=90,
                        va="center", ha="center", fontweight="bold", fontsize=11)
            ax.set_xticks([])
            ax.set_yticks([])
            cb = fig.colorbar(image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.035)
            cb.set_label("count" if variable == "N_data" else group.units, fontsize=8)
            cb.ax.tick_params(labelsize=7)

    fig.suptitle(
        f"{EXPERIMENT_LABEL}: observation support and mean fields, {PERIOD_LABEL}\n"
        "Nobs is from the DA run; obs and forecast means are from the paired OL monitor baseline",
        y=0.995,
        fontsize=14,
    )
    fig.subplots_adjust(top=0.955, bottom=0.025, left=0.055, right=0.985, hspace=0.25, wspace=0.08)
    path = OUT / f"{EXPERIMENT_KEY}_nobs_omean_fmean_maps_5x3.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")

    table = pd.DataFrame(summary_rows)
    table_path = OUT / f"{EXPERIMENT_KEY}_nobs_omean_fmean_maps_summary.csv"
    table.to_csv(table_path, index=False)
    return fig, path, table, table_path

fig, input_map_path, input_map_summary, input_map_summary_path = plot_observation_input_maps()
display(input_map_summary)
plt.show()
plt.close(fig)
print("wrote", input_map_path.relative_to(PROJECT))
print("wrote", input_map_summary_path.relative_to(PROJECT))


In [ ]:
def read_example_obs_month(path: Path = EXAMPLE_OBS) -> pd.DataFrame:
    rows = []
    for file_path in sorted(path.glob("cygnss_l1_ddm3x5_crop_scalar_m36_*_all_cyg.nc4")):
        date = datetime.strptime(file_path.name.split("_m36_")[1].split("_all_cyg")[0], "%Y%m%d")
        with Dataset(file_path) as ds:
            n_obs = len(ds.dimensions["obs"])
            frame = pd.DataFrame({
                "date": [date] * n_obs,
                "file": [file_path.name] * n_obs,
                "obs_id": np.asarray(ds["obs_id"][:], dtype=int),
                "sc_num": np.asarray(ds["sc_num"][:], dtype=int),
                "sp_lon": np.asarray(ds["sp_lon"][:], dtype=float),
                "sp_lat": np.asarray(ds["sp_lat"][:], dtype=float),
                "observed_y_db": np.asarray(ds["observed_y_db"][:], dtype=float),
                "ddm_snr_db": np.asarray(ds["ddm_snr_db"][:], dtype=float),
                "sp_inc_angle": np.asarray(ds["sp_inc_angle"][:], dtype=float),
                "coefficient_weighted_opacity": np.asarray(ds["coefficient_weighted_opacity"][:], dtype=float),
                "ddm_time_utc": [str(value) for value in ds["ddm_time_utc"][:]],
            })
            rows.append(frame)
    if not rows:
        raise FileNotFoundError(f"No CYGNSS L1 example obs files found under {path}")
    return pd.concat(rows, ignore_index=True)


def binned_obs_field(obs: pd.DataFrame, value: str | None, resolution: float = 0.25) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    lon_edges = np.arange(MAP_EXTENT[0], MAP_EXTENT[1] + resolution, resolution)
    lat_edges = np.arange(MAP_EXTENT[2], MAP_EXTENT[3] + resolution, resolution)
    lon_idx = np.digitize(obs["sp_lon"], lon_edges) - 1
    lat_idx = np.digitize(obs["sp_lat"], lat_edges) - 1
    inside = (lon_idx >= 0) & (lon_idx < len(lon_edges) - 1) & (lat_idx >= 0) & (lat_idx < len(lat_edges) - 1)
    shape = (len(lat_edges) - 1, len(lon_edges) - 1)
    counts = np.zeros(shape, dtype=float)
    sums = np.zeros(shape, dtype=float)
    for i, j, keep in zip(lon_idx, lat_idx, inside):
        if keep:
            counts[j, i] += 1.0
    if value is None:
        field = counts
    else:
        vals = obs[value].to_numpy(dtype=float)
        for i, j, keep, val in zip(lon_idx, lat_idx, inside, vals):
            if keep and np.isfinite(val):
                sums[j, i] += val
        field = np.divide(sums, counts, out=np.full(shape, np.nan), where=counts > 0)
    field = np.where(counts > 0, field, np.nan)
    return lon_edges, lat_edges, field


def plot_example_obs_maps():
    obs = read_example_obs_month()
    fields = (
        (None, "Observation count", "magma", "count"),
        ("observed_y_db", "Mean observed y", "viridis", "dB"),
        ("ddm_snr_db", "Mean DDM SNR", "plasma", "dB"),
        ("sp_inc_angle", "Mean incidence angle", "cividis", "degree"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.8), subplot_kw={"projection": ccrs.PlateCarree()})
    summary_rows = []
    for ax, (variable, title, cmap, label) in zip(axes.ravel(), fields):
        base_map(ax)
        lon_edges, lat_edges, field = binned_obs_field(obs, variable)
        norm = norm_from_values([field], lower=2, upper=98) if variable is not None else norm_from_values([field], lower=0, upper=99)
        mesh = ax.pcolormesh(
            lon_edges, lat_edges, field, cmap=cmap, norm=norm, shading="flat",
            transform=ccrs.PlateCarree(), zorder=2,
        )
        ax.scatter(
            obs["sp_lon"], obs["sp_lat"], s=3.5, c="k", alpha=0.18, linewidths=0,
            transform=ccrs.PlateCarree(), zorder=3,
        )
        ax.set_title(title, loc="left", fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])
        cb = fig.colorbar(mesh, ax=ax, orientation="horizontal", fraction=0.052, pad=0.035)
        cb.set_label(label, fontsize=8.5)
        cb.ax.tick_params(labelsize=8)
        values = field[np.isfinite(field)]
        summary_rows.append({
            "variable": "N_obs" if variable is None else variable,
            "finite_bins": int(values.size),
            "mean": float(np.nanmean(values)),
            "median": float(np.nanmedian(values)),
            "min": float(np.nanmin(values)),
            "max": float(np.nanmax(values)),
        })

    n_files = obs["file"].nunique()
    fig.suptitle(
        f"CYGNSS L1 example observations, June 2020 ({n_files} daily files, {len(obs):,} obs)\n"
        "Specular-point locations shown as black dots; gridded fields use 0.25 degree bins",
        fontsize=13.5,
        y=0.99,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    path = OUT / "dense075_coh05_example_obs_june2020_maps_2x2.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    summary = pd.DataFrame(summary_rows)
    summary.insert(0, "obs_files", n_files)
    summary.insert(1, "obs_count", len(obs))
    summary_path = OUT / "dense075_coh05_example_obs_june2020_maps_summary.csv"
    summary.to_csv(summary_path, index=False)
    return obs, fig, path, summary, summary_path

example_obs, fig, example_obs_map_path, example_obs_map_summary, example_obs_map_summary_path = plot_example_obs_maps()
display(example_obs_map_summary)
plt.show()
plt.close(fig)
print("wrote", example_obs_map_path.relative_to(PROJECT))
print("wrote", example_obs_map_summary_path.relative_to(PROJECT))


In [ ]:
def add_assimilation_window(obs: pd.DataFrame) -> pd.DataFrame:
    out = obs.copy()
    times = pd.to_datetime(out["ddm_time_utc"], format="mixed", utc=True)
    out["ddm_timestamp"] = times
    out["assim_time_utc"] = times.dt.round("3h")
    out["assim_date"] = out["assim_time_utc"].dt.strftime("%Y-%m-%d")
    out["assim_cycle"] = out["assim_time_utc"].dt.strftime("%Hz")
    return out


def plot_example_obs_assimilation_windows(date: str, value: str = "observed_y_db"):
    obs = add_assimilation_window(read_example_obs_month())
    selected = obs[obs["assim_date"] == date].copy()
    if selected.empty:
        raise ValueError(f"No observations found for assimilation-window date {date}")

    cycles = [f"{hour:02d}z" for hour in range(0, 24, 3)]
    fig, axes = plt.subplots(2, 4, figsize=(15.2, 7.6), subplot_kw={"projection": ccrs.PlateCarree()})
    norm = norm_from_values([selected[value].to_numpy(dtype=float)], lower=2, upper=98)
    last_image = None
    summary_rows = []

    for ax, cycle in zip(axes.ravel(), cycles):
        base_map(ax)
        this = selected[selected["assim_cycle"] == cycle]
        if not this.empty:
            last_image = ax.scatter(
                this["sp_lon"], this["sp_lat"], c=this[value], s=28, marker="o",
                linewidths=0.25, edgecolors="0.15", cmap="viridis", norm=norm,
                transform=ccrs.PlateCarree(), zorder=3,
            )
        ax.set_title(f"{date} {cycle}  n={len(this)}", loc="left", fontweight="bold", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        vals = this[value].to_numpy(dtype=float) if not this.empty else np.asarray([], dtype=float)
        summary_rows.append({
            "assim_date": date,
            "assim_cycle": cycle,
            "obs_count": int(len(this)),
            f"{value}_mean": float(np.nanmean(vals)) if vals.size else np.nan,
            f"{value}_median": float(np.nanmedian(vals)) if vals.size else np.nan,
            "snr_mean_db": float(np.nanmean(this["ddm_snr_db"])) if len(this) else np.nan,
            "incidence_angle_mean_deg": float(np.nanmean(this["sp_inc_angle"])) if len(this) else np.nan,
        })

    if last_image is not None:
        cb = fig.colorbar(last_image, ax=axes.ravel().tolist(), orientation="horizontal", fraction=0.045, pad=0.055)
        cb.set_label("observed_y_db (dB)" if value == "observed_y_db" else value, fontsize=9.5)
    fig.suptitle(
        f"CYGNSS L1 example observations by 3-hour assimilation window, {date}\n"
        "Window label is nearest 3-hour GEOSldas cycle time; points are observation specular locations",
        fontsize=13.5,
        y=0.99,
    )
    fig.subplots_adjust(top=0.88, bottom=0.13, left=0.035, right=0.99, hspace=0.18, wspace=0.05)

    tag = date.replace("-", "")
    path = OUT / f"dense075_coh05_example_obs_assim_windows_{tag}_2x4.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    summary = pd.DataFrame(summary_rows)
    summary_path = OUT / f"dense075_coh05_example_obs_assim_windows_{tag}_summary.csv"
    summary.to_csv(summary_path, index=False)
    return selected, fig, path, summary, summary_path

# Set the individual assimilation-window date to plot here.
plot_date = "2020-06-12"

window_obs, fig, window_map_path, window_map_summary, window_map_summary_path = plot_example_obs_assimilation_windows(plot_date)
display(window_map_summary)
plt.show()
plt.close(fig)
print("wrote", window_map_path.relative_to(PROJECT))
print("wrote", window_map_summary_path.relative_to(PROJECT))


In [ ]:
M36_SCALE = 36032.220840584
M36_NCOL = 964
M36_NROW = 406
SLIDE_MAP_FIGSIZE = (13.5, 9.5)
SLIDE_MAP_DPI = 180
SLIDE_MAP_SUBPLOTS = {"left": 0.035, "right": 0.99, "bottom": 0.12, "top": 0.88}


def easev2_m36_grid_edges() -> tuple[np.ndarray, np.ndarray]:
    transformer = Transformer.from_crs(
        CRS.from_proj4(
            "+proj=cea +lat_ts=30 +lon_0=0 +x_0=0 +y_0=0 "
            "+ellps=WGS84 +datum=WGS84 +units=m"
        ),
        CRS.from_epsg(4326),
        always_xy=True,
    )
    x = (np.arange(M36_NCOL + 1) - 0.5 - (M36_NCOL - 1) / 2.0) * M36_SCALE
    y = ((M36_NROW - 1) / 2.0 - (np.arange(M36_NROW + 1) - 0.5)) * M36_SCALE
    lon_edges, _ = transformer.transform(x, np.zeros_like(x))
    _, lat_edges = transformer.transform(np.zeros_like(y), y)
    return lon_edges, lat_edges


M36_LON_EDGES, M36_LAT_EDGES = easev2_m36_grid_edges()


def example_obs_file_for_date(date: str) -> Path:
    stamp = date.replace("-", "")
    path = EXAMPLE_OBS / f"cygnss_l1_ddm3x5_crop_scalar_m36_{stamp}_all_cyg.nc4"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def coefficient_field_for_obs(file_path: Path, obs_position: int) -> tuple[np.ma.MaskedArray, dict[str, float | int | str]]:
    with Dataset(file_path) as ds:
        start = int(ds["tile_start"][obs_position])
        count = int(ds["tile_count"][obs_position])
        slc = slice(start, start + count)
        raw = np.asarray(ds["coefficient"][slc], dtype=float)
        tile_i = np.asarray(ds["tile_ig"][slc], dtype=int)
        tile_j = np.asarray(ds["tile_jg"][slc], dtype=int)
        total = float(np.nansum(raw))
        weights = np.divide(raw, total, out=np.full(raw.shape, np.nan), where=total > 0)

        field = np.full((M36_NROW, M36_NCOL), np.nan, dtype=float)
        valid = (
            np.isfinite(weights)
            & (weights > 0)
            & (tile_i >= 0) & (tile_i < M36_NCOL)
            & (tile_j >= 0) & (tile_j < M36_NROW)
        )
        field[tile_j[valid], tile_i[valid]] = weights[valid]
        meta = {
            "obs_position": int(obs_position),
            "obs_id": int(ds["obs_id"][obs_position]),
            "time": str(ds["ddm_time_utc"][obs_position]),
            "sc_num": int(ds["sc_num"][obs_position]),
            "tile_count": int(count),
            "raw_coefficient_sum": total,
            "normalized_sum": float(np.nansum(weights)),
            "observed_y_db": float(ds["observed_y_db"][obs_position]),
            "snr_db": float(ds["ddm_snr_db"][obs_position]),
            "sp_lon": float(ds["sp_lon"][obs_position]),
            "sp_lat": float(ds["sp_lat"][obs_position]),
        }
    return np.ma.masked_invalid(field), meta


def obs_positions_for_window(date: str, cycle: str, max_obs: int | None = None) -> list[int]:
    obs = add_assimilation_window(read_example_obs_month())
    selected = obs[(obs["assim_date"] == date) & (obs["assim_cycle"] == cycle)].copy()
    if selected.empty:
        raise ValueError(f"No observations found for {date} {cycle}")
    file_path = example_obs_file_for_date(date)
    with Dataset(file_path) as ds:
        obs_ids = np.asarray(ds["obs_id"][:], dtype=int)
        tile_counts = np.asarray(ds["tile_count"][:], dtype=int)
    selected_ids = set(selected["obs_id"].astype(int))
    positions = [pos for pos, obs_id in enumerate(obs_ids) if int(obs_id) in selected_ids]
    positions = sorted(positions, key=lambda pos: tile_counts[pos], reverse=True)
    return positions if max_obs is None else positions[:max_obs]


def coefficient_norm(fields: list[np.ma.MaskedArray]) -> LogNorm:
    finite_weights = np.concatenate([field.compressed() for field in fields if field.count()])
    vmin = max(float(np.nanpercentile(finite_weights, 5)), 1.0e-8)
    vmax = min(max(float(np.nanmax(finite_weights)), vmin * 10), 1.0)
    return LogNorm(vmin=vmin, vmax=vmax)


def plot_coefficient_weight_maps(date: str, cycle: str, max_obs: int = 4):
    file_path = example_obs_file_for_date(date)
    positions = obs_positions_for_window(date, cycle, max_obs=max_obs)
    fields = []
    metas = []
    for pos in positions:
        field, meta = coefficient_field_for_obs(file_path, pos)
        fields.append(field)
        metas.append(meta)
    norm = coefficient_norm(fields)

    ncols = 2 if len(fields) <= 4 else 3
    nrows = int(np.ceil(len(fields) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5.8 * ncols, 5.35 * nrows),
        subplot_kw={"projection": ccrs.PlateCarree()}, squeeze=False,
    )
    last_image = None
    summary_rows = []
    for ax, field, meta in zip(axes.ravel(), fields, metas):
        base_map(ax)
        last_image = ax.pcolormesh(
            M36_LON_EDGES, M36_LAT_EDGES, field,
            cmap="magma", norm=norm, shading="flat",
            transform=ccrs.PlateCarree(), zorder=2,
        )
        ax.scatter(
            [meta["sp_lon"]], [meta["sp_lat"]], marker="*", s=80, c="#2b83ba",
            edgecolors="white", linewidths=0.8, transform=ccrs.PlateCarree(), zorder=4,
        )
        ax.set_title(
            f"obs_id {meta['obs_id']}  tiles={meta['tile_count']}  sum={meta['normalized_sum']:.3f}\n"
            f"{meta['time']}  y={meta['observed_y_db']:.1f} dB  SNR={meta['snr_db']:.1f} dB",
            loc="left", fontweight="bold", fontsize=9.5,
        )
        ax.set_xticks([])
        ax.set_yticks([])
        row = dict(meta)
        row["date"] = date
        row["assim_cycle"] = cycle
        summary_rows.append(row)
    for ax in axes.ravel()[len(fields):]:
        ax.axis("off")
    if last_image is not None:
        cb = fig.colorbar(
            last_image, ax=axes.ravel().tolist(), orientation="horizontal",
            fraction=0.034, pad=0.075, aspect=45,
        )
        cb.set_label("normalized tile coefficient, raw coefficient / sum(raw coefficient)", fontsize=9.5)
    fig.suptitle(
        f"CYGNSS L1 tile-coefficient footprints, {date} {cycle}\n"
        "Each panel is one observation; full M36 tiles are shown with pcolormesh; blue star is the specular point",
        fontsize=13.5,
        y=0.99,
    )
    fig.subplots_adjust(top=0.86, bottom=0.16, left=0.035, right=0.99, hspace=0.28, wspace=0.055)
    tag = f"{date.replace('-', '')}_{cycle}"
    path = OUT / f"dense075_coh05_example_obs_coefficients_{tag}_pcolormesh.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    summary = pd.DataFrame(summary_rows)
    summary_path = OUT / f"dense075_coh05_example_obs_coefficients_{tag}_summary.csv"
    summary.to_csv(summary_path, index=False)
    return fig, path, summary, summary_path


def plot_all_coefficient_weight_overlay(date: str, cycle: str):
    file_path = example_obs_file_for_date(date)
    positions = obs_positions_for_window(date, cycle, max_obs=None)
    fields = []
    metas = []
    for pos in positions:
        field, meta = coefficient_field_for_obs(file_path, pos)
        fields.append(field)
        metas.append(meta)
    norm = coefficient_norm(fields)

    fig = plt.figure(figsize=SLIDE_MAP_FIGSIZE)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    base_map(ax)
    last_image = None
    for index, field in enumerate(fields):
        last_image = ax.pcolormesh(
            M36_LON_EDGES, M36_LAT_EDGES, field,
            cmap="magma", norm=norm, shading="flat", alpha=0.55,
            transform=ccrs.PlateCarree(), zorder=2 + index * 0.001,
        )
    ax.scatter(
        [meta["sp_lon"] for meta in metas], [meta["sp_lat"] for meta in metas],
        marker="*", s=28, c="#2b83ba", edgecolors="white", linewidths=0.45,
        transform=ccrs.PlateCarree(), zorder=5,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    if last_image is not None:
        cb = fig.colorbar(last_image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.04)
        cb.set_label("normalized tile coefficient for each observation; overplots are not combined", fontsize=10)
    fig.suptitle(
        f"All CYGNSS L1 tile-coefficient footprints overplotted, {date} {cycle}\n"
        f"{len(fields)} observations drawn sequentially with alpha=0.55; overlaps are intentionally not aggregated",
        fontsize=14,
        y=0.96,
    )
    tag = f"{date.replace('-', '')}_{cycle}"
    fig.subplots_adjust(**SLIDE_MAP_SUBPLOTS)
    path = OUT / f"dense075_coh05_example_obs_coefficients_{tag}_all_overlay.png"
    fig.savefig(path, dpi=SLIDE_MAP_DPI)
    summary = pd.DataFrame(metas)
    summary.insert(0, "date", date)
    summary.insert(1, "assim_cycle", cycle)
    summary_path = OUT / f"dense075_coh05_example_obs_coefficients_{tag}_all_overlay_summary.csv"
    summary.to_csv(summary_path, index=False)
    return fig, path, summary, summary_path


# Set the date/cycle for individual coefficient-footprint maps here.
coefficient_plot_date = "2020-06-01"
coefficient_plot_cycle = "06z"
coefficient_max_obs = 6

fig, coefficient_map_path, coefficient_map_summary, coefficient_map_summary_path = plot_coefficient_weight_maps(
    coefficient_plot_date,
    coefficient_plot_cycle,
    max_obs=coefficient_max_obs,
)
display(coefficient_map_summary)
plt.show()
plt.close(fig)
print("wrote", coefficient_map_path.relative_to(PROJECT))
print("wrote", coefficient_map_summary_path.relative_to(PROJECT))

fig, coefficient_overlay_path, coefficient_overlay_summary, coefficient_overlay_summary_path = plot_all_coefficient_weight_overlay(
    coefficient_plot_date,
    coefficient_plot_cycle,
)
display(coefficient_overlay_summary.head())
plt.show()
plt.close(fig)
print("wrote", coefficient_overlay_path.relative_to(PROJECT))
print("wrote", coefficient_overlay_summary_path.relative_to(PROJECT))


In [ ]:
EXAMPLE_OFA = PROJECT / "example_ofa" / "M06"
EXAMPLE_CAT = PROJECT / "example_cat" / "M06"


def cycle_stamp(cycle: str) -> str:
    if cycle.endswith("z") and len(cycle) == 3:
        return f"{cycle[:2]}00z"
    if cycle.endswith("00z"):
        return cycle
    raise ValueError(f"Expected cycle like '06z' or '0600z', got {cycle!r}")


def dense_example_file(root: Path, product: str, date: str, cycle: str) -> Path:
    stamp = date.replace("-", "")
    path = root / f"DAv8_M36_AZ_paired_cygl1_dense075_coh05.{product}.{stamp}_{cycle_stamp(cycle)}.nc4"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def read_cygl1_ofa(date: str, cycle: str) -> pd.DataFrame:
    path = dense_example_file(EXAMPLE_OFA, "ens_avg.ldas_ObsFcstAna", date, cycle)
    with Dataset(path) as ds:
        species = np.asarray(ds["species"][:], dtype=int)
        keep = species == 13
        frame = pd.DataFrame({
            "ofa_index": np.arange(species.size)[keep],
            "lon": np.asarray(ds["lon"][:], dtype=float)[keep],
            "lat": np.asarray(ds["lat"][:], dtype=float)[keep],
            "obs": np.asarray(ds["obs"][:], dtype=float)[keep],
            "fcst": np.asarray(ds["fcst"][:], dtype=float)[keep],
            "ana": np.asarray(ds["ana"][:], dtype=float)[keep],
            "assim_flag": np.asarray(ds["assim_flag"][:], dtype=int)[keep],
        })
    frame["omf"] = frame["obs"] - frame["fcst"]
    frame["oma"] = frame["obs"] - frame["ana"]
    return frame


def matched_cygl1_footprints(date: str, cycle: str, tolerance_deg: float = 1.0e-5) -> pd.DataFrame:
    file_path = example_obs_file_for_date(date)
    positions = obs_positions_for_window(date, cycle, max_obs=None)
    ofa = read_cygl1_ofa(date, cycle)
    rows = []
    used: set[int] = set()
    with Dataset(file_path) as ds:
        for pos in positions:
            sp_lon = float(ds["sp_lon"][pos])
            sp_lat = float(ds["sp_lat"][pos])
            if ofa.empty:
                continue
            distance = np.hypot(ofa["lon"].to_numpy() - sp_lon, ofa["lat"].to_numpy() - sp_lat)
            order = np.argsort(distance)
            match_index = None
            for idx in order:
                ofa_index = int(ofa.iloc[idx]["ofa_index"])
                if ofa_index not in used and distance[idx] <= tolerance_deg:
                    match_index = idx
                    used.add(ofa_index)
                    break
            if match_index is None:
                continue
            matched = ofa.iloc[match_index]
            start = int(ds["tile_start"][pos])
            count = int(ds["tile_count"][pos])
            raw = np.asarray(ds["coefficient"][start:start + count], dtype=float)
            total = float(np.nansum(raw))
            rows.append({
                "obs_position": int(pos),
                "obs_id": int(ds["obs_id"][pos]),
                "time": str(ds["ddm_time_utc"][pos]),
                "sp_lon": sp_lon,
                "sp_lat": sp_lat,
                "tile_count": count,
                "raw_coefficient_sum": total,
                "normalized_sum": float(np.nansum(raw / total)) if total > 0 else np.nan,
                "ofa_index": int(matched["ofa_index"]),
                "obs": float(matched["obs"]),
                "fcst": float(matched["fcst"]),
                "ana": float(matched["ana"]),
                "omf": float(matched["omf"]),
                "oma": float(matched["oma"]),
            })
    return pd.DataFrame(rows)


def slide_base_map(title: str):
    fig = plt.figure(figsize=SLIDE_MAP_FIGSIZE)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    base_map(ax)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, loc="left", fontweight="bold", fontsize=14, pad=8)
    return fig, ax


def save_slide_map(fig, path: Path):
    fig.subplots_adjust(**SLIDE_MAP_SUBPLOTS)
    fig.savefig(path, dpi=SLIDE_MAP_DPI)
    return path


def plot_ofa_point_slide_map(date: str, cycle: str, variable: str, norm: Normalize, label: str, cmap: str = "viridis"):
    ofa = read_cygl1_ofa(date, cycle)
    if ofa.empty:
        raise ValueError(f"No CYGL1 OFA rows for {date} {cycle}")
    fig, ax = slide_base_map(f"CYGNSS L1 OFA {variable.upper()} values, {date} {cycle}")
    image = ax.scatter(
        ofa["lon"], ofa["lat"], c=ofa[variable], s=44, marker="o",
        cmap=cmap, norm=norm, edgecolors="0.15", linewidths=0.35,
        transform=ccrs.PlateCarree(), zorder=4,
    )
    cb = fig.colorbar(image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.04)
    cb.set_label(label, fontsize=10)
    tag = f"{date.replace('-', '')}_{cycle}"
    path = OUT / f"dense075_coh05_example_slide_{variable}_{tag}.png"
    save_slide_map(fig, path)
    summary = pd.DataFrame([{
        "date": date,
        "assim_cycle": cycle,
        "product": "ens_avg.ldas_ObsFcstAna",
        "variable": variable,
        "n_points": int(len(ofa)),
        "mean": float(np.nanmean(ofa[variable])),
        "median": float(np.nanmedian(ofa[variable])),
        "min": float(np.nanmin(ofa[variable])),
        "max": float(np.nanmax(ofa[variable])),
        "path": str(path.relative_to(PROJECT)),
    }])
    return fig, path, summary


def m36_tile_field_from_file(path: Path, variable: str) -> tuple[np.ma.MaskedArray, pd.DataFrame]:
    with Dataset(path) as ds:
        ig = np.asarray(ds["IG"][:], dtype=int)
        jg = np.asarray(ds["JG"][:], dtype=int)
        data = np.asarray(ds[variable][0, :], dtype=float)
        tile_lat = np.asarray(ds["lat"][:], dtype=float)
        tile_lon = np.asarray(ds["lon"][:], dtype=float)
    field = np.full((M36_NROW, M36_NCOL), np.nan, dtype=float)
    valid = np.isfinite(data) & (ig >= 0) & (ig < M36_NCOL) & (jg >= 0) & (jg < M36_NROW)
    field[jg[valid], ig[valid]] = data[valid]
    table = pd.DataFrame({"IG": ig, "JG": jg, "lat": tile_lat, "lon": tile_lon, variable: data})
    return np.ma.masked_invalid(field), table


def symmetric_norm(values: np.ndarray, percentile: float = 99.0) -> TwoSlopeNorm:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    limit = float(np.nanpercentile(np.abs(finite), percentile)) if finite.size else 1.0
    limit = max(limit, 1.0e-12)
    return TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)


def add_cygl1_obs_stars(ax, date: str, cycle: str):
    ofa = read_cygl1_ofa(date, cycle)
    if ofa.empty:
        return None
    return ax.scatter(
        ofa["lon"], ofa["lat"], marker="*", s=22, c="white",
        edgecolors="0.15", linewidths=0.35, transform=ccrs.PlateCarree(), zorder=5,
    )


def plot_tile_slide_map(date: str, cycle: str, product: str, variable: str, title: str, label: str, path_token: str):
    path = dense_example_file(EXAMPLE_CAT, product, date, cycle)
    field, table = m36_tile_field_from_file(path, variable)
    fig, ax = slide_base_map(title)
    norm = symmetric_norm(field.compressed())
    image = ax.pcolormesh(
        M36_LON_EDGES, M36_LAT_EDGES, field,
        cmap="RdBu", norm=norm, shading="flat", transform=ccrs.PlateCarree(), zorder=2,
    )
    add_cygl1_obs_stars(ax, date, cycle)
    cb = fig.colorbar(image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.04)
    cb.set_label(label, fontsize=10)
    tag = f"{date.replace('-', '')}_{cycle}"
    out_path = OUT / f"dense075_coh05_example_slide_{path_token}_{tag}.png"
    save_slide_map(fig, out_path)
    values = table[variable].to_numpy(dtype=float)
    summary = pd.DataFrame([{
        "date": date,
        "assim_cycle": cycle,
        "product": product,
        "variable": variable,
        "finite_tiles": int(np.isfinite(values).sum()),
        "nonzero_tiles": int(np.sum(np.isfinite(values) & (np.abs(values) > 0))),
        "mean": float(np.nanmean(values)),
        "median": float(np.nanmedian(values)),
        "min": float(np.nanmin(values)),
        "max": float(np.nanmax(values)),
        "path": str(out_path.relative_to(PROJECT)),
    }])
    return fig, out_path, summary


def plot_tile_difference_slide_map(date: str, cycle: str, ana_variable: str, fcst_variable: str, title: str, label: str, path_token: str):
    path = dense_example_file(EXAMPLE_CAT, "inst3_1d_lndfcstana_Nt", date, cycle)
    with Dataset(path) as ds:
        ig = np.asarray(ds["IG"][:], dtype=int)
        jg = np.asarray(ds["JG"][:], dtype=int)
        tile_lat = np.asarray(ds["lat"][:], dtype=float)
        tile_lon = np.asarray(ds["lon"][:], dtype=float)
        diff = np.asarray(ds[ana_variable][0, :] - ds[fcst_variable][0, :], dtype=float)
    field = np.full((M36_NROW, M36_NCOL), np.nan, dtype=float)
    valid = np.isfinite(diff) & (ig >= 0) & (ig < M36_NCOL) & (jg >= 0) & (jg < M36_NROW)
    field[jg[valid], ig[valid]] = diff[valid]
    fig, ax = slide_base_map(title)
    norm = symmetric_norm(diff)
    image = ax.pcolormesh(
        M36_LON_EDGES, M36_LAT_EDGES, np.ma.masked_invalid(field),
        cmap="RdBu", norm=norm, shading="flat", transform=ccrs.PlateCarree(), zorder=2,
    )
    add_cygl1_obs_stars(ax, date, cycle)
    cb = fig.colorbar(image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.04)
    cb.set_label(label, fontsize=10)
    tag = f"{date.replace('-', '')}_{cycle}"
    out_path = OUT / f"dense075_coh05_example_slide_{path_token}_{tag}.png"
    save_slide_map(fig, out_path)
    summary = pd.DataFrame([{
        "date": date,
        "assim_cycle": cycle,
        "product": "inst3_1d_lndfcstana_Nt",
        "variable": f"{ana_variable}_minus_{fcst_variable}",
        "finite_tiles": int(np.isfinite(diff).sum()),
        "nonzero_tiles": int(np.sum(np.isfinite(diff) & (np.abs(diff) > 0))),
        "mean": float(np.nanmean(diff)),
        "median": float(np.nanmedian(diff)),
        "min": float(np.nanmin(diff)),
        "max": float(np.nanmax(diff)),
        "path": str(out_path.relative_to(PROJECT)),
    }])
    return fig, out_path, summary


# Slide-aligned single-map sequence for the 2020-06-01 06z example window.
slide_map_date = "2020-06-01"
slide_map_cycle = "06z"

ofa_for_norm = read_cygl1_ofa(slide_map_date, slide_map_cycle)
obs_ana_norm = norm_from_values([ofa_for_norm["obs"].to_numpy(), ofa_for_norm["ana"].to_numpy()], lower=2, upper=98)
omf_norm = symmetric_norm(ofa_for_norm["omf"].to_numpy())

slide_outputs = []
for variable, label, norm, cmap in (
    ("obs", "CYGNSS L1 observed y (dB)", obs_ana_norm, "viridis"),
    ("ana", "CYGNSS L1 analysis H(x) (dB)", obs_ana_norm, "viridis"),
    ("omf", "CYGNSS L1 Obs - Forecast (dB)", omf_norm, "RdBu_r"),
):
    fig, path, summary = plot_ofa_point_slide_map(slide_map_date, slide_map_cycle, variable, norm, label, cmap=cmap)
    slide_outputs.append({"figure": path, "summary": summary, "variable": variable})
    display(summary)
    plt.show()
    plt.close(fig)
    print("wrote", path.relative_to(PROJECT))

for product, variable, title, label, token in (
    ("catch_progn_incr", "SRFEXC_INCR", f"Surface excess increment, {slide_map_date} {slide_map_cycle}", "SRFEXC_INCR (kg m-2)", "srfexc_incr"),
    ("catch_progn_incr", "RZEXC_INCR", f"Root-zone excess increment, {slide_map_date} {slide_map_cycle}", "RZEXC_INCR (kg m-2)", "rzexc_incr"),
):
    fig, path, summary = plot_tile_slide_map(slide_map_date, slide_map_cycle, product, variable, title, label, token)
    slide_outputs.append({"figure": path, "summary": summary, "variable": variable})
    display(summary)
    plt.show()
    plt.close(fig)
    print("wrote", path.relative_to(PROJECT))

for ana_variable, fcst_variable, title, label, token in (
    ("SFMC_ANA", "SFMC_FCST", f"Surface soil moisture analysis minus forecast, {slide_map_date} {slide_map_cycle}", "SFMC_ANA - SFMC_FCST (m3 m-3)", "sfmc_ana_minus_fcst"),
    ("RZMC_ANA", "RZMC_FCST", f"Root-zone soil moisture analysis minus forecast, {slide_map_date} {slide_map_cycle}", "RZMC_ANA - RZMC_FCST (m3 m-3)", "rzmc_ana_minus_fcst"),
):
    fig, path, summary = plot_tile_difference_slide_map(slide_map_date, slide_map_cycle, ana_variable, fcst_variable, title, label, token)
    slide_outputs.append({"figure": path, "summary": summary, "variable": f"{ana_variable}_minus_{fcst_variable}"})
    display(summary)
    plt.show()
    plt.close(fig)
    print("wrote", path.relative_to(PROJECT))

slide_summary_rows = []
for item in slide_outputs:
    summary = item["summary"]
    if len(summary) == 1 and "path" in summary.columns:
        slide_summary_rows.append(summary.iloc[0].to_dict())
    else:
        variable = item.get("variable", item["figure"].stem.replace("dense075_coh05_example_slide_", ""))
        values = summary[variable].to_numpy(dtype=float) if variable in summary.columns else np.asarray([])
        slide_summary_rows.append({
            "date": slide_map_date,
            "assim_cycle": slide_map_cycle,
            "product": "ens_avg.ldas_ObsFcstAna",
            "variable": variable,
            "n_points": int(len(summary)),
            "mean": float(np.nanmean(values)) if values.size else np.nan,
            "median": float(np.nanmedian(values)) if values.size else np.nan,
            "min": float(np.nanmin(values)) if values.size else np.nan,
            "max": float(np.nanmax(values)) if values.size else np.nan,
            "path": str(item["figure"].relative_to(PROJECT)),
        })
slide_summary = pd.DataFrame(slide_summary_rows)
slide_summary_path = OUT / f"dense075_coh05_example_slide_maps_{slide_map_date.replace('-', '')}_{slide_map_cycle}_summary.csv"
slide_summary.to_csv(slide_summary_path, index=False)
display(slide_summary)
print("wrote", slide_summary_path.relative_to(PROJECT))

Saved outputs:

- `output/thinning_expts/figures/dense075_coh05_omf_stdv_maps_5x3.png`
- `output/thinning_expts/figures/dense075_coh05_omf_stdv_percent_map_summary.csv`
- `output/thinning_expts/figures/dense075_coh05_monthly_omf_stdv_5x2.png`
- `output/thinning_expts/figures/dense075_coh05_monthly_omf_stdv_summary.csv`
- `output/thinning_expts/figures/dense075_coh05_monthly_omf_mean_5x2.png`
- `output/thinning_expts/figures/dense075_coh05_monthly_omf_mean_summary.csv`
- `output/thinning_expts/figures/dense075_coh05_nobs_omean_fmean_maps_5x3.png`
- `output/thinning_expts/figures/dense075_coh05_nobs_omean_fmean_maps_summary.csv`
- `output/thinning_expts/figures/dense075_coh05_example_obs_june2020_maps_2x2.png`
- `output/thinning_expts/figures/dense075_coh05_example_obs_june2020_maps_summary.csv`
- `output/thinning_expts/figures/dense075_coh05_example_obs_assim_windows_20200601_2x4.png`
- `output/thinning_expts/figures/dense075_coh05_example_obs_assim_windows_20200601_summary.csv`
- `output/thinning_expts/figures/dense075_coh05_example_obs_coefficients_20200601_06z_pcolormesh.png`
- `output/thinning_expts/figures/dense075_coh05_example_obs_coefficients_20200601_06z_summary.csv`
- `output/thinning_expts/figures/dense075_coh05_example_obs_coefficients_20200601_06z_all_overlay.png`
- `output/thinning_expts/figures/dense075_coh05_example_obs_coefficients_20200601_06z_all_overlay_summary.csv`

- `output/thinning_expts/figures/dense075_coh05_example_slide_obs_20200601_06z.png`
- `output/thinning_expts/figures/dense075_coh05_example_slide_ana_20200601_06z.png`
- `output/thinning_expts/figures/dense075_coh05_example_slide_omf_20200601_06z.png`
- `output/thinning_expts/figures/dense075_coh05_example_slide_srfexc_incr_20200601_06z.png`
- `output/thinning_expts/figures/dense075_coh05_example_slide_rzexc_incr_20200601_06z.png`
- `output/thinning_expts/figures/dense075_coh05_example_slide_sfmc_ana_minus_fcst_20200601_06z.png`
- `output/thinning_expts/figures/dense075_coh05_example_slide_rzmc_ana_minus_fcst_20200601_06z.png`
- `output/thinning_expts/figures/dense075_coh05_example_slide_maps_20200601_06z_summary.csv`
